# Clasificador de perros y gatos

En este proyecto se construye un clasificador de imágenes de perros y gatos siguiendo el enunciado oficial de 4Geeks Academy. El trabajo se estructura en tres fases:

1. **CNN simple propia** como línea base.
2. **Arquitectura VGG** exacta del enunciado, entrenada desde cero con `ModelCheckpoint` y `EarlyStopping`.
3. **Transfer learning** con `MobileNetV2` como experimento complementario.

> **Nota sobre el hardware:** Este notebook se ejecuta en GPU (Windows + RTX 3060 Ti). Se usa `ImageDataGenerator` con `flow_from_directory()` para cargar las imágenes por lotes sin saturar la RAM. El entrenamiento de VGG es mucho mas rapido; `EarlyStopping` evita epochs innecesarias.

> **Nota sobre `.fit()` vs `.fit_generator()`:** El enunciado menciona `fit_generator`, pero este método está **deprecado** desde Keras 2.4. En versiones modernas, `.fit()` acepta directamente generadores (`Sequence`) y hereda todas las funcionalidades de `fit_generator`, incluyendo callbacks, validación por lotes y multiproceso. Se usa `.fit()` en todo el notebook.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from pathlib import Path
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
from sklearn.metrics import confusion_matrix
import warnings
warnings.filterwarnings("ignore")

print(f"TensorFlow version: {tf.__version__}")
print(f"Keras version: {keras.__version__}")


## 2. Dataset y estrategia de carga

El dataset completo tiene ~25.000 imágenes. Dado que la RAM es limitada, se usa `ImageDataGenerator` con `flow_from_directory()` para cargar las imágenes por lotes durante el entrenamiento.

Las imágenes originales (`dogs-vs-cats/train/`) están todas en una sola carpeta. Para poder usar `flow_from_directory` se reorganizaron en:

```
data/processed/dogs-vs-cats/
  train/
    cat/   (10.000 imágenes)
    dog/   (10.000 imágenes)
  test/
    cat/   (2.500 imágenes)
    dog/   (2.500 imágenes)
```

El split es 80 % train / 20 % test. La carpeta `validation` se obtiene como un 20 % del split de train mediante `validation_split` en el generador.

> El enunciado pide imágenes de 200×200, pero la arquitectura VGG del Paso 3 usa 224×224. Se unifica todo a **224×224** para no redimensionar dos veces.


In [ ]:
# Rutas
processed_dir = Path("../data/processed/dogs-vs-cats")
train_dir = processed_dir / "train"
test_dir = processed_dir / "test"

# Parámetros unificados para todo el notebook
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

print(f"Tamaño de imagen: {IMG_SIZE}")
print(f"Batch size: {BATCH_SIZE}")


## 3. Exploración visual inicial


### 3.1. Ejemplos de perros


In [ ]:
dog_images = sorted((train_dir / "dog").glob("*.jpg"))

plt.figure(figsize=(8, 8))
for i, img_path in enumerate(dog_images[:9]):
    img = plt.imread(str(img_path))
    plt.subplot(3, 3, i + 1)
    plt.imshow(img)
    plt.axis("off")
    plt.title("Perro")
plt.suptitle("Primeras 9 imágenes de perros", fontsize=14)
plt.tight_layout()
plt.show()


### 3.2. Ejemplos de gatos


In [ ]:
cat_images = sorted((train_dir / "cat").glob("*.jpg"))

plt.figure(figsize=(8, 8))
for i, img_path in enumerate(cat_images[:9]):
    img = plt.imread(str(img_path))
    plt.subplot(3, 3, i + 1)
    plt.imshow(img)
    plt.axis("off")
    plt.title("Gato")
plt.suptitle("Primeras 9 imágenes de gatos", fontsize=14)
plt.tight_layout()
plt.show()


## 4. Generadores de Keras (`ImageDataGenerator`)

Se crean dos generadores:
- **Entrenamiento:** con augmentación ligera (rescale + flip horizontal) y `validation_split=0.2` para obtener validación.
- **Test:** solo rescale, sin augmentación.

`flow_from_directory` infiere las etiquetas automáticamente desde los nombres de las subcarpetas (`cat`, `dog`) y las codifica como **categóricas** (one-hot) o **binarias** según `class_mode`. Para la arquitectura VGG del enunciado (`Dense(2, softmax)`) usamos `class_mode='categorical'`. Para la CNN simple y MobileNetV2 (`binary_crossentropy`) usamos `class_mode='binary'`.

> Nota: se usa `subset='training'` y `subset='validation'` para que Keras divida automáticamente el 20 % de validación a partir de `train_dir`.


In [ ]:
# Generador de entrenamiento (con split de validación)
train_datagen = ImageDataGenerator(
    rescale=1.0 / 255,
    horizontal_flip=True,
    validation_split=0.2,
)

# Generador de test (sin augmentación)
test_datagen = ImageDataGenerator(rescale=1.0 / 255)

# --- Flujo para CNN simple y MobileNetV2 (binario) ---
train_generator_binary = train_datagen.flow_from_directory(
    train_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="binary",
    subset="training",
    seed=42,
)

val_generator_binary = train_datagen.flow_from_directory(
    train_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="binary",
    subset="validation",
    seed=42,
)

test_generator_binary = test_datagen.flow_from_directory(
    test_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="binary",
    shuffle=False,
)

# --- Flujo para VGG (categorical, 2 clases con softmax) ---
train_generator_cat = train_datagen.flow_from_directory(
    train_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    subset="training",
    seed=42,
)

val_generator_cat = train_datagen.flow_from_directory(
    train_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    subset="validation",
    seed=42,
)

test_generator_cat = test_datagen.flow_from_directory(
    test_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=False,
)

print(f"\nTrain samples (binary): {train_generator_binary.samples}")
print(f"Validation samples (binary): {val_generator_binary.samples}")
print(f"Test samples (binary): {test_generator_binary.samples}")
print(f"Classes: {train_generator_binary.class_indices}")


## 5. Fase 1. CNN simple propia

Modelo base con pocas capas convolucionales para establecer una línea base antes de la arquitectura VGG.


In [ ]:
simple_model = keras.Sequential([
    layers.Input(shape=(224, 224, 3)),
    layers.Conv2D(32, (3, 3), activation="relu"),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(64, (3, 3), activation="relu"),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(128, (3, 3), activation="relu"),
    layers.MaxPooling2D((2, 2)),
    layers.Flatten(),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.5),
    layers.Dense(1, activation="sigmoid"),
], name="cnn_simple")

simple_model.summary()


## 6. Fase 1. Entrenamiento del modelo base


In [ ]:
simple_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"],
)

# Steps per epoch = ceil(samples / batch_size)
steps_train = int(np.ceil(train_generator_binary.samples / BATCH_SIZE))
steps_val = int(np.ceil(val_generator_binary.samples / BATCH_SIZE))

simple_history = simple_model.fit(
    train_generator_binary,
    steps_per_epoch=steps_train,
    epochs=5,
    validation_data=val_generator_binary,
    validation_steps=steps_val,
    verbose=1,
)


## 7. Fase 1. Evaluación del modelo base


In [ ]:
steps_test = int(np.ceil(test_generator_binary.samples / BATCH_SIZE))
simple_test_loss, simple_test_acc = simple_model.evaluate(test_generator_binary, steps=steps_test)
print(f"CNN simple - Test loss: {simple_test_loss:.4f}")
print(f"CNN simple - Test accuracy: {simple_test_acc:.4f}")

# Gráficas
history_df = pd.DataFrame(simple_history.history)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.lineplot(data=history_df[["accuracy", "val_accuracy"]], ax=axes[0])
axes[0].set_title("Accuracy")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Accuracy")

sns.lineplot(data=history_df[["loss", "val_loss"]], ax=axes[1])
axes[1].set_title("Loss")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Loss")

plt.tight_layout()
plt.show()


## 8. Fase 2. Arquitectura VGG del enunciado

Se construye la arquitectura **exacta** que proporciona el enunciado del proyecto:

- Bloques de convolución con filtros crecientes: 64 → 128 → 256 → 512 → 512
- Cada bloque tiene 2 o 3 capas `Conv2D` seguidas de `MaxPool2D`
- Capas densas finales: 4096 → 4096 → 2 (softmax)
- Input: 224 × 224 × 3

> Nota: el enunciado usa `Dense(2, softmax)` con `categorical_crossentropy`, por lo que las etiquetas deben estar en formato one-hot (lo gestiona `class_mode='categorical'` del generador).


In [ ]:
vgg_model = keras.Sequential([
    layers.Input(shape=(224, 224, 3)),
    
    # Bloque 1: 64 filtros
    layers.Conv2D(64, (3, 3), padding="same", activation="relu"),
    layers.Conv2D(64, (3, 3), padding="same", activation="relu"),
    layers.MaxPooling2D((2, 2), strides=(2, 2)),
    
    # Bloque 2: 128 filtros
    layers.Conv2D(128, (3, 3), padding="same", activation="relu"),
    layers.Conv2D(128, (3, 3), padding="same", activation="relu"),
    layers.MaxPooling2D((2, 2), strides=(2, 2)),
    
    # Bloque 3: 256 filtros
    layers.Conv2D(256, (3, 3), padding="same", activation="relu"),
    layers.Conv2D(256, (3, 3), padding="same", activation="relu"),
    layers.Conv2D(256, (3, 3), padding="same", activation="relu"),
    layers.MaxPooling2D((2, 2), strides=(2, 2)),
    
    # Bloque 4: 512 filtros
    layers.Conv2D(512, (3, 3), padding="same", activation="relu"),
    layers.Conv2D(512, (3, 3), padding="same", activation="relu"),
    layers.Conv2D(512, (3, 3), padding="same", activation="relu"),
    layers.MaxPooling2D((2, 2), strides=(2, 2)),
    
    # Bloque 5: 512 filtros
    layers.Conv2D(512, (3, 3), padding="same", activation="relu"),
    layers.Conv2D(512, (3, 3), padding="same", activation="relu"),
    layers.Conv2D(512, (3, 3), padding="same", activation="relu"),
    layers.MaxPooling2D((2, 2), strides=(2, 2)),
    
    # Capas densas
    layers.Flatten(),
    layers.Dense(4096, activation="relu"),
    layers.Dense(4096, activation="relu"),
    layers.Dense(2, activation="softmax"),
], name="vgg_enunciado")

vgg_model.summary()


## 9. Fase 2. Callbacks: `ModelCheckpoint` y `EarlyStopping`

El enunciado pide usar estos callbacks en el entrenamiento:

- **`ModelCheckpoint`**: guarda el mejor modelo según `val_loss` en cada epoch.
- **`EarlyStopping`**: detiene el entrenamiento si `val_loss` no mejora tras 3 epochs consecutivas, restaurando los mejores pesos encontrados.

> Esto evita sobreajuste y desperdiciar tiempo de entrenamiento en GPU (Windows + RTX 3060 Ti).


In [ ]:
# Asegurar que existe la carpeta models
models_dir = Path("../models")
models_dir.mkdir(parents=True, exist_ok=True)

checkpoint = ModelCheckpoint(
    filepath=str(models_dir / "best_vgg_model.keras"),
    monitor="val_loss",
    save_best_only=True,
    verbose=1,
)

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True,
    verbose=1,
)

print(f"Checkpoint guardará en: {models_dir / 'best_vgg_model.keras'}")


## 10. Fase 2. Entrenamiento del modelo VGG


In [ ]:
vgg_model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

steps_train_vgg = int(np.ceil(train_generator_cat.samples / BATCH_SIZE))
steps_val_vgg = int(np.ceil(val_generator_cat.samples / BATCH_SIZE))

vgg_history = vgg_model.fit(
    train_generator_cat,
    steps_per_epoch=steps_train_vgg,
    epochs=10,
    validation_data=val_generator_cat,
    validation_steps=steps_val_vgg,
    callbacks=[checkpoint, early_stop],
    verbose=1,
)


## 11. Fase 2. Evaluación del VGG en memoria


In [ ]:
steps_test_vgg = int(np.ceil(test_generator_cat.samples / BATCH_SIZE))
vgg_test_loss, vgg_test_acc = vgg_model.evaluate(test_generator_cat, steps=steps_test_vgg)
print(f"VGG (en memoria) - Test loss: {vgg_test_loss:.4f}")
print(f"VGG (en memoria) - Test accuracy: {vgg_test_acc:.4f}")

# Gráficas
history_df_vgg = pd.DataFrame(vgg_history.history)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.lineplot(data=history_df_vgg[["accuracy", "val_accuracy"]], ax=axes[0])
axes[0].set_title("VGG - Accuracy")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Accuracy")

sns.lineplot(data=history_df_vgg[["loss", "val_loss"]], ax=axes[1])
axes[1].set_title("VGG - Loss")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Loss")

plt.tight_layout()
plt.show()


## 12. Fase 2. Carga del mejor modelo desde disco y evaluación

El enunciado pide explícitamente: *"Carga el mejor modelo de los anteriores y utiliza el conjunto de test para hacer predicciones."*

Se carga el `.keras` guardado por `ModelCheckpoint` y se evalúa sobre el conjunto de test.


In [ ]:
best_vgg_model = keras.models.load_model(str(models_dir / "best_vgg_model.keras"))

best_vgg_test_loss, best_vgg_test_acc = best_vgg_model.evaluate(test_generator_cat, steps=steps_test_vgg)
print(f"VGG (mejor guardado) - Test loss: {best_vgg_test_loss:.4f}")
print(f"VGG (mejor guardado) - Test accuracy: {best_vgg_test_acc:.4f}")


## 13. Fase 3. Transfer learning con `MobileNetV2`

Como experimento complementario, se usa `MobileNetV2` preentrenada en ImageNet. Se adapta a 224×224 (su tamaño nativo) y se añade una cabeza de clasificación binaria.

Esta fase preserva el trabajo pedagógico previo sobre transfer learning.


In [ ]:
base_model = keras.applications.MobileNetV2(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3),
)
base_model.trainable = False

transfer_model = keras.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.2),
    layers.Dense(1, activation="sigmoid"),
], name="mobilenetv2_transfer")

transfer_model.summary()


## 14. Fase 3. Entrenamiento del modelo de transfer learning


In [ ]:
transfer_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"],
)

transfer_history = transfer_model.fit(
    train_generator_binary,
    steps_per_epoch=steps_train,
    epochs=5,
    validation_data=val_generator_binary,
    validation_steps=steps_val,
    verbose=1,
)


## 15. Fase 3. Evaluación del transfer learning


In [ ]:
transfer_test_loss, transfer_test_acc = transfer_model.evaluate(test_generator_binary, steps=steps_test)
print(f"Transfer learning - Test loss: {transfer_test_loss:.4f}")
print(f"Transfer learning - Test accuracy: {transfer_test_acc:.4f}")


## 16. Comparativa final de los tres modelos


In [ ]:
results = pd.DataFrame({
    "Modelo": ["CNN simple", "VGG (mejor guardado)", "Transfer learning"],
    "Test loss": [simple_test_loss, best_vgg_test_loss, transfer_test_loss],
    "Test accuracy": [simple_test_acc, best_vgg_test_acc, transfer_test_acc],
})

print(results.to_string(index=False))

# Gráfica comparativa
fig, ax = plt.subplots(figsize=(8, 5))
x = np.arange(len(results))
width = 0.35
bars1 = ax.bar(x - width/2, results["Test loss"], width, label="Test loss", color="coral")
bars2 = ax.bar(x + width/2, results["Test accuracy"], width, label="Test accuracy", color="skyblue")
ax.set_xticks(x)
ax.set_xticklabels(results["Modelo"])
ax.set_ylabel("Valor")
ax.set_title("Comparativa de modelos")
ax.legend()
plt.ylim(0, 1.2)
plt.tight_layout()
plt.show()


## 17. Guardado de modelos

Se guardan ambos modelos entrenados en `../models/`:
1. El mejor VGG (ya guardado por `ModelCheckpoint` en `best_vgg_model.keras`).
2. El modelo de transfer learning.


In [ ]:
# Guardar modelo de transfer learning
transfer_path = models_dir / "transfer_learning_mobilenetv2_224.keras"
transfer_model.save(transfer_path)
print(f"Modelo de transfer learning guardado en: {transfer_path}")

# Verificar archivos en ../models/
print("\nArchivos en ../models/:")
for f in sorted(models_dir.glob("*.keras")):
    size_mb = f.stat().st_size / (1024 * 1024)
    print(f"  {f.name}: {size_mb:.1f} MB")


## 18. Conclusiones

1. **CNN simple**: establece una línea base rápida de entrenar. Con pocas epochs sobre el dataset completo, sirve para validar que el pipeline de datos funciona.

2. **VGG exacta del enunciado**: es la arquitectura requerida por el proyecto. Tiene ~134 millones de parámetros y entrena desde cero. Los callbacks `ModelCheckpoint` y `EarlyStopping` son esenciales para evitar sobreajuste y no desperdiciar tiempo de GPU.

3. **Transfer learning con MobileNetV2**: como experimento complementario, demuestra que reutilizar una red preentrenada en ImageNet acelera drásticamente la convergencia. Es el enfoque de producción real para este tipo de problema.

4. **Diferencia entre `.fit()` y `.fit_generator()`**: el enunciado menciona `fit_generator`, pero este método fue deprecado en Keras 2.4. En versiones modernas, `.fit()` acepta generadores (`Sequence`) directamente, heredando todas las capacidades de `fit_generator` (multiproceso, callbacks, validación por lotes). Este notebook usa `.fit()` como reemplazo directo y válido.
